In [ ]:
#| default_exp vision

# vision

> Read pixels, shape them the way a model wants them, and turn raw output tensors back into labels and boxes.

Everything here is numpy and Pillow. No runtime is imported, so this module works whether or not
LiteRT, ONNX Runtime or Core ML is installed, and the runtimes in `anya.onnx` / `anya.litert` /
`anya.apple` share one preprocessor and one decoder rather than each growing their own.

In [ ]:
#| export
from __future__ import annotations
import io, json, mimetypes
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from fastcore.all import L

In [ ]:
#| hide
from fastcore.test import *

## What is this file

`media_kind` decides image, audio or video from the suffix first and the magic bytes second, so a
`.jpeg` with the wrong name still loads.

In [ ]:
#| export
IMG_EXTS = set('.jpg .jpeg .png .bmp .gif .webp .tif .tiff .ppm .pgm'.split())
AUD_EXTS = set('.wav .mp3 .flac .ogg .oga .m4a .aac .opus'.split())
VID_EXTS = set('.mp4 .mov .avi .mkv .webm .m4v .mpg .mpeg'.split())
MEDIA_EXTS = IMG_EXTS | AUD_EXTS | VID_EXTS

_MAGIC = ((b'\xff\xd8\xff', 'image'), (b'\x89PNG', 'image'), (b'GIF8', 'image'),
          (b'BM', 'image'), (b'RIFF', 'audio'), (b'\x00\x00\x00', 'video'), (b'OggS', 'audio'),
          (b'fLaC', 'audio'), (b'ID3', 'audio'))

def media_kind(o) -> str|None:
    "`'image'`, `'audio'`, `'video'` or None for a path, bytes, array or PIL image."
    if isinstance(o, np.ndarray): return 'image' if o.ndim in (2, 3) else 'audio'
    if hasattr(o, 'mode') and hasattr(o, 'size'): return 'image'          # a PIL image, undeclared
    if isinstance(o, (bytes, bytearray)):
        for sig, k in _MAGIC:
            if bytes(o[:8]).startswith(sig): return k
        return None
    p = Path(str(o))
    ext = p.suffix.lower()
    if ext in IMG_EXTS: return 'image'
    if ext in AUD_EXTS: return 'audio'
    if ext in VID_EXTS: return 'video'
    if (mt := mimetypes.guess_type(str(p))[0]): return mt.split('/')[0] if mt.split('/')[0] in ('image','audio','video') else None
    return None

In [ ]:
#| hide
test_eq(media_kind('bird.JPG'), 'image')
test_eq(media_kind(Path('clip.wav')), 'audio')
test_eq(media_kind('a.mp4'), 'video')
test_eq(media_kind('notes.txt'), None)
test_eq(media_kind(np.zeros((4,4,3), np.uint8)), 'image')

## Loading

`load_image` takes a path, a URL-free `bytes` blob, a PIL image or an array already, and always
returns `uint8 HWC RGB`. One shape in the rest of the library means the preprocessor never has to
ask where a picture came from.

In [ ]:
#| export
def load_image(o,                 # path, str, bytes, PIL image, or HWC/HW array
               mode:str='RGB'     # PIL mode to convert to
              ) -> np.ndarray:
    'Read anything image-shaped into a `uint8` HWC array in `mode`.'
    from PIL import Image, ImageOps
    if isinstance(o, np.ndarray):
        a = o if o.ndim == 3 else np.repeat(o[..., None], 3, -1)
        return a.astype(np.uint8) if a.dtype != np.uint8 else a
    im = o if hasattr(o, 'convert') else Image.open(io.BytesIO(o) if isinstance(o, (bytes, bytearray)) else str(o))
    im = ImageOps.exif_transpose(im) or im       # a phone photo is stored rotated and tagged
    return np.asarray(im.convert(mode), dtype=np.uint8)

def img_size(o) -> tuple:
    "`(height, width)` of an image without decoding the pixels where Pillow can avoid it."
    from PIL import Image
    if isinstance(o, np.ndarray): return o.shape[:2]
    if hasattr(o, 'size'): return (o.size[1], o.size[0])
    with Image.open(io.BytesIO(o) if isinstance(o, (bytes, bytearray)) else str(o)) as im: return (im.size[1], im.size[0])

In [ ]:
#| hide
from PIL import Image
from tempfile import mkdtemp
_td = Path(mkdtemp())
_img = (np.arange(32*24*3, dtype=np.uint8).reshape(24,32,3))
Image.fromarray(_img).save(_td/'t.png')
test_eq(load_image(_td/'t.png').shape, (24,32,3))
test_eq(load_image(_td/'t.png').dtype, np.uint8)
test_eq(img_size(_td/'t.png'), (24,32))
test_eq(load_image(np.zeros((5,5), np.uint8)).shape, (5,5,3))     # greyscale is broadcast, not rejected

In [ ]:
#| export
def load_audio(o,               # path, str, or bytes of an audio file
               sr:int=16000     # target sample rate
              ) -> tuple:
    'Decode audio to mono `float32` at `sr`, returning `(samples, sr)`.'
    try: import soundfile as sf
    except ImportError as e: raise ImportError("anya needs soundfile to read audio: pip install 'anya[audio]'") from e
    x, in_sr = sf.read(io.BytesIO(o) if isinstance(o, (bytes, bytearray)) else str(o), dtype='float32', always_2d=True)
    x = x.mean(1)
    if in_sr != sr:                                  # linear resample: good enough for feature extractors
        n = int(round(len(x) * sr / in_sr))
        x = np.interp(np.linspace(0, len(x)-1, n), np.arange(len(x)), x).astype(np.float32)
    return x, sr

def video_frames(o,                    # path or str of a video file
                 every:float=1.0,      # seconds between sampled frames
                 max_frames:int=None   # stop after this many
                ):
    'Yield `(timestamp, uint8 HWC array)` every `every` seconds of a video.'
    try: import av
    except ImportError as e: raise ImportError("anya needs PyAV to read video: pip install 'anya[video]'") from e
    with av.open(str(o)) as c:
        st = c.streams.video[0]
        st.thread_type = 'AUTO'
        nxt, n = 0.0, 0
        for f in c.decode(st):
            t = float(f.pts * st.time_base) if f.pts is not None else nxt
            if t + 1e-6 < nxt: continue
            yield t, f.to_ndarray(format='rgb24')
            n, nxt = n+1, t + every
            if max_frames and n >= max_frames: return

## Preprocessing

`Prep` is everything a model needs done to a picture before it sees it, in one comparable object.
The runtimes fill it in from the model's own signature and metadata, so the caller does not write
a normalisation line and does not get it wrong.

`resize='letterbox'` keeps aspect ratio and records the padding in the returned meta, which is what
`scale_boxes` needs to put detections back on the original picture.

In [ ]:
#| export
BILINEAR, BICUBIC = 2, 3          # PIL's resample ids, and what a preprocessor config puts in `resample`

@dataclass
class Prep:
    'How to turn a `uint8` HWC image into the exact tensor one model expects.'
    size:tuple=None                  # (h, w) the model wants; None leaves the image alone
    layout:str='nhwc'                # 'nhwc' or 'nchw'
    dtype:str='float32'              # tensor dtype the model declares
    scale:float=1/255                # multiply pixels by this first
    mean:tuple=(0., 0., 0.)          # then subtract, per channel
    std:tuple=(1., 1., 1.)           # then divide, per channel
    resize:str='stretch'             # 'stretch', 'letterbox', or 'center_crop'
    crop_pct:float=None              # with 'center_crop', the fraction of the short side kept
    resample:int=BILINEAR            # PIL filter id, the way `preprocessor_config.json` writes it
    quant:tuple=None                 # (scale, zero_point) of a quantised input tensor
    bgr:bool=False                   # channel order the model was trained on

    def __repr__(self):
        x = ''.join(f', {k}={v}' for k, v in (('crop_pct', self.crop_pct), ('quant', self.quant)) if v)
        r = '' if self.resample == BILINEAR else f', resample={self.resample}'
        return f'Prep({self.size}, {self.layout}, {self.dtype}, resize={self.resize}{x}{r})'

@dataclass
class AudioPrep:
    'How to turn decoded audio into the exact tensor one model expects.'
    sr:int=16000                     # sample rate the model was trained at
    samples:int=None                 # fixed input length; shorter is padded, longer is trimmed
    dtype:str='float32'
    layout:str='nt'                  # 'nt' (batch, samples) or 't' (samples only, no batch axis)

    def __repr__(self): return f'AudioPrep({self.sr}Hz, {self.samples or "any"} samples)'

IMAGENET = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))   # torchvision's, the most common in the wild

In [ ]:
#| export
def _pil_resize(a, size, resample=BILINEAR):
    from PIL import Image
    return np.asarray(Image.fromarray(a).resize((size[1], size[0]), resample), dtype=np.uint8)

def letterbox(a,                    # uint8 HWC image
              size:tuple,           # (h, w) to fit into
              color:int=114,        # pad value, YOLO's grey
              resample=BILINEAR
             ) -> tuple:
    'Resize `a` into `size` keeping aspect ratio, padding the rest; returns `(image, meta)`.'
    h, w = a.shape[:2]; th, tw = size
    r = min(th/h, tw/w)
    nh, nw = max(1, int(round(h*r))), max(1, int(round(w*r)))
    out = np.full((th, tw, a.shape[2]), color, np.uint8)
    top, left = (th-nh)//2, (tw-nw)//2
    out[top:top+nh, left:left+nw] = _pil_resize(a, (nh, nw), resample)
    return out, dict(ratio=r, pad=(left, top), orig=(h, w), size=(th, tw))

def center_crop(a,                  # uint8 HWC image
                size:tuple,         # (h, w) to end up with
                pct:float=None,     # fraction of the short side to keep, timm's crop_pct
                resample=BILINEAR
               ) -> tuple:
    'Scale the short side to `size/pct` then crop `size` out of the centre; returns `(image, meta)`.'
    h, w = a.shape[:2]; th, tw = size
    sh, sw = (max(th, round(th/pct)), max(tw, round(tw/pct))) if pct else (th, tw)
    r = max(sh/h, sw/w)
    nh, nw = max(sh, int(round(h*r))), max(sw, int(round(w*r)))
    b = _pil_resize(a, (nh, nw), resample)
    top, left = (nh-th)//2, (nw-tw)//2
    return b[top:top+th, left:left+tw], dict(ratio=r, pad=(-left, -top), orig=(h, w), size=(th, tw))

def fit(a, size:tuple, mode:str='stretch', pct:float=None, resample=BILINEAR) -> tuple:
    'Resize `a` to `size` by `mode`; returns `(image, meta)` where meta un-maps coordinates.'
    if size is None: return a, dict(ratio=1.0, pad=(0,0), orig=a.shape[:2], size=a.shape[:2])
    if mode == 'letterbox': return letterbox(a, size, resample=resample)
    if mode == 'center_crop': return center_crop(a, size, pct, resample)
    h, w = a.shape[:2]
    return (_pil_resize(a, size, resample),
            dict(ratio=(size[0]/h, size[1]/w), pad=(0,0), orig=(h, w), size=tuple(size)))

In [ ]:
#| hide
_a = np.zeros((10, 40, 3), np.uint8)
_b, _m = letterbox(_a, (20, 20))
test_eq(_b.shape, (20,20,3)); test_eq(_m['ratio'], 0.5); test_eq(_m['pad'], (0, 7))
test_eq(_b[0,0].tolist(), [114]*3)          # the pad is grey, the image is black
test_eq(_b[10,10].tolist(), [0]*3)
_c, _ = center_crop(_a, (8, 8)); test_eq(_c.shape, (8,8,3))
test_eq(fit(_a, (5,5))[0].shape, (5,5,3))
# crop_pct=0.875 is timm's: a 224 crop out of a 256 short side, so the resize is bigger than the crop
_c, _m = center_crop(np.zeros((480, 640, 3), np.uint8), (224, 224), 0.875)
test_eq(_c.shape, (224,224,3)); test_close(_m['ratio'], 256/480); test_eq(_m['pad'], (-58, -16))

In [ ]:
#| export
def apply_prep(a,          # uint8 HWC image
               p:Prep      # what the model wants
              ) -> tuple:
    'Apply `p` to one image; returns `(tensor without batch axis, meta)`.'
    x, meta = fit(a, p.size, p.resize, p.crop_pct, p.resample)
    if p.bgr: x = x[..., ::-1]
    if p.quant:                                     # a quantised model wants the raw integer grid
        qs, zp = p.quant
        v = (x.astype(np.float32) * p.scale - np.array(p.mean, np.float32)) / np.array(p.std, np.float32)
        x = np.clip(np.round(v / qs + zp), *(_dtype_range(p.dtype))).astype(p.dtype)
    elif p.dtype.startswith(('uint', 'int')): x = x.astype(p.dtype)
    else:
        x = x.astype(np.float32) * p.scale
        x = (x - np.array(p.mean, np.float32)) / np.array(p.std, np.float32)
        x = x.astype(p.dtype)
    if p.layout == 'nchw': x = np.transpose(x, (2, 0, 1))
    return np.ascontiguousarray(x), meta

def _dtype_range(dt):
    i = np.iinfo(np.dtype(dt)); return i.min, i.max

def apply_audio_prep(x,             # float32 mono samples
                     p:AudioPrep    # what the model wants
                    ) -> tuple:
    'Pad or trim `x` to the length the model declares; returns `(tensor, meta)`.'
    x = np.asarray(x, np.float32)
    n0 = len(x)
    if p.samples:
        x = x[:p.samples] if n0 >= p.samples else np.pad(x, (0, p.samples-n0))
    return x.astype(p.dtype), dict(samples=n0, seconds=round(n0/p.sr, 3), sr=p.sr)

def batch(xs,                  # images (uint8 HWC) or waveforms, matching `p`
          p                    # a `Prep` or an `AudioPrep`
         ) -> tuple:
    'Preprocess `xs` and stack them; returns `(batched tensor, list of meta)`.'
    f = apply_audio_prep if isinstance(p, AudioPrep) else apply_prep
    ys, ms = zip(*[f(a, p) for a in xs])
    out = np.stack(ys)
    if isinstance(p, AudioPrep) and p.layout == 't': out = out[0] if len(ys) == 1 else out
    return out, list(ms)

In [ ]:
#| hide
_p = Prep(size=(4,4), dtype='float32', scale=1/255, mean=IMAGENET[0], std=IMAGENET[1], layout='nchw')
_x, _mt = apply_prep(np.full((8,8,3), 255, np.uint8), _p)
test_eq(_x.shape, (3,4,4))
test_close(_x[0,0,0], (1-0.485)/0.229)
_q = Prep(size=(4,4), dtype='uint8', quant=(1/255, 0))       # a quantised model gets integers back
test_eq(apply_prep(np.full((8,8,3), 255, np.uint8), _q)[0].max(), 255)
test_eq(batch([np.zeros((6,6,3), np.uint8)]*3, Prep(size=(2,2)))[0].shape, (3,2,2,3))
_ap = AudioPrep(sr=16000, samples=8)
_x, _am = apply_audio_prep(np.ones(4, np.float32), _ap)
test_eq(len(_x), 8); test_eq(_x[5], 0.); test_eq(_am['samples'], 4)      # padded, and the true length recorded
test_eq(apply_audio_prep(np.ones(20, np.float32), _ap)[0].shape, (8,))   # and trimmed the other way
test_eq(batch([np.ones(9, np.float32)], _ap)[0].shape, (1, 8))

## Decoding

One decoder per task, taking raw output arrays and returning plain Python. These are the parts that
are wrong in most hand-written glue: a softmax applied twice, boxes left in letterbox coordinates,
an off-by-one in a label file that has a background class at index 0.

In [ ]:
#| export
def softmax(x, axis=-1):
    'Numerically stable softmax.'
    e = np.exp(np.asarray(x, np.float32) - np.max(x, axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, np.float32)))

def is_prob(x) -> bool:
    'Does `x` already look like a probability vector (0..1 and summing to 1)?'
    x = np.asarray(x, np.float32)
    return bool(x.min() >= -1e-4 and x.max() <= 1+1e-4 and abs(float(x.sum())-1) < 1e-2)

def label_at(labels, i:int) -> str:
    "Label `i`, or `'class_{i}'` when no labels were supplied."
    if labels is None or i >= len(labels): return f'class_{i}'
    return labels[i]

In [ ]:
#| export
def decode_classify(out,                # raw model output, (n_classes,) or (1, n_classes)
                    labels=None,        # class names, index-aligned
                    topk:int=5,         # how many to keep
                    multi_label:bool=False   # sigmoid per class instead of softmax over classes
                   ) -> list:
    'Turn a classifier output into a ranked list of `{label, score, index}`.'
    v = np.asarray(out, np.float32).reshape(-1)
    p = sigmoid(v) if multi_label else (v if is_prob(v) else softmax(v))
    idx = np.argsort(-p)[:max(1, topk)]
    return [dict(label=label_at(labels, int(i)), score=round(float(p[i]), 6), index=int(i)) for i in idx]

In [ ]:
#| hide
test_eq(decode_classify([0., 10., 0.], ['a','b','c'], topk=1)[0]['label'], 'b')
test_close(decode_classify([0.1, 0.9], topk=2)[0]['score'], 0.9)   # already a probability, left alone
test_eq(decode_classify([0., 10., 0.], ['a','b'], topk=3)[2]['label'], 'class_2')  # short label file

In [ ]:
#| export
def xywh2xyxy(b):
    'Centre-form boxes to corner-form.'
    b = np.asarray(b, np.float32); o = np.empty_like(b)
    o[..., 0] = b[..., 0] - b[..., 2]/2; o[..., 1] = b[..., 1] - b[..., 3]/2
    o[..., 2] = b[..., 0] + b[..., 2]/2; o[..., 3] = b[..., 1] + b[..., 3]/2
    return o

def nms(boxes, scores, iou:float=0.45, max_det:int=300) -> list:
    'Greedy non-maximum suppression; returns the indices to keep.'
    b, s = np.asarray(boxes, np.float32), np.asarray(scores, np.float32)
    if not len(b): return []
    x1, y1, x2, y2 = b[:,0], b[:,1], b[:,2], b[:,3]
    area = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    order, keep = np.argsort(-s), []
    while len(order) and len(keep) < max_det:
        i = order[0]; keep.append(int(i))
        xx1, yy1 = np.maximum(x1[i], x1[order[1:]]), np.maximum(y1[i], y1[order[1:]])
        xx2, yy2 = np.minimum(x2[i], x2[order[1:]]), np.minimum(y2[i], y2[order[1:]])
        inter = np.clip(xx2-xx1, 0, None) * np.clip(yy2-yy1, 0, None)
        ovr = inter / np.maximum(area[i] + area[order[1:]] - inter, 1e-9)
        order = order[1:][ovr <= iou]
    return keep

In [ ]:
#| hide
test_eq(nms([[0,0,10,10],[1,1,11,11],[50,50,60,60]], [0.9,0.8,0.7]), [0,2])
test_eq(len(nms([], [])), 0)
test_close(xywh2xyxy([5,5,4,4]), [3,3,7,7])

In [ ]:
#| export
def scale_boxes(boxes,          # xyxy in model input pixels
                meta:dict,      # the meta `fit` returned
                clip:bool=True  # keep boxes inside the picture
               ) -> np.ndarray:
    'Map boxes from preprocessed coordinates back onto the original image.'
    b = np.asarray(boxes, np.float32).copy()
    if not len(b): return b
    (px, py), r, (h, w) = meta.get('pad', (0,0)), meta.get('ratio', 1.0), meta.get('orig', (None, None))
    b[:, [0,2]] -= px; b[:, [1,3]] -= py
    if isinstance(r, (tuple, list)): b[:, [0,2]] /= r[1]; b[:, [1,3]] /= r[0]
    else: b /= r
    if clip and h: b[:, [0,2]] = b[:, [0,2]].clip(0, w); b[:, [1,3]] = b[:, [1,3]].clip(0, h)
    return b

In [ ]:
#| export
def decode_yolo(out,                 # (1, 4+nc, N) or (1, N, 4+nc), optionally with objectness
                labels=None,
                conf:float=0.25,
                iou:float=0.45,
                meta:dict=None,      # from `fit`, to put boxes back on the original image
                max_det:int=300,
                normalized:bool=None   # boxes are 0..1 rather than input pixels; None reads it off their range
               ) -> list:
    'Decode a YOLO-family output into `{label, score, box, index}` dicts with xyxy boxes.'
    a = np.asarray(out, np.float32)
    if a.ndim == 3: a = a[0]
    if a.shape[0] < a.shape[1]: a = a.T          # (4+nc, N) -> (N, 4+nc)
    nc = a.shape[1] - 4
    box, cls = a[:, :4], a[:, 4:]
    # the same (1, 84, 8400) signature ships both units: two of three yolo exports tested are normalised
    if normalized is None: normalized = bool(len(box)) and float(box.max()) <= 2.0
    obj = (len(labels) == nc-1) if labels is not None and len(labels) in (nc, nc-1) else _looks_objectness(cls)
    if obj and nc > 1: cls = cls[:, 1:] * cls[:, :1]      # v5/v7 put an objectness column first
    if cls.max(initial=0.) > 1.0 + 1e-3: cls = sigmoid(cls)
    score, idx = cls.max(1), cls.argmax(1)
    m = score >= conf
    box, score, idx = xywh2xyxy(box[m]), score[m], idx[m]
    if normalized and meta: box *= np.array([meta['size'][1], meta['size'][0]]*2, np.float32)
    keep = nms(box, score, iou, max_det)
    box = scale_boxes(box[keep], meta) if meta else box[keep]
    return [dict(label=label_at(labels, int(idx[k])), score=round(float(score[k]), 6), index=int(idx[k]),
                 box=[round(float(v), 2) for v in box[j]]) for j, k in enumerate(keep)]

def _looks_objectness(cls) -> bool:
    "Is column 0 an objectness score rather than a class? Only when it dominates in every row."
    if cls.shape[1] < 2 or not len(cls): return False
    return bool((cls[:, 0] >= cls[:, 1:].max(1)).all())

In [ ]:
#| hide
# one box at (50,50) 20x20, class 1 of 3, in a 100x100 letterboxed input over a 200x100 original
_o = np.zeros((1, 7, 20), np.float32)
_o[0, :4, 0] = [50, 50, 20, 20]; _o[0, 5, 0] = 0.9
_m = dict(ratio=0.5, pad=(0, 25), orig=(100, 200), size=(100, 100))
_d = decode_yolo(_o, ['a','b','c'], conf=0.5, meta=_m)
test_eq(len(_d), 1); test_eq(_d[0]['label'], 'b')
test_close(_d[0]['box'], [80., 30., 120., 70.])       # un-padded and un-scaled back to the original
# the same head with 0..1 boxes: 0.5,0.5,0.2,0.2 of a 100x100 input is the very same object
_n = np.zeros((1, 7, 20), np.float32); _n[0, :4, 0] = [.5, .5, .2, .2]; _n[0, 5, 0] = 0.9
test_close(decode_yolo(_n, ['a','b','c'], conf=0.5, meta=_m)[0]['box'], [80., 30., 120., 70.])

In [ ]:
#| export
def decode_ssd(outs,             # the four tensors of a TFLite Detection PostProcess graph
               labels=None,
               conf:float=0.25,
               meta:dict=None,
               offset:int=0      # add to every class index (some label files lead with background)
              ) -> list:
    'Decode boxes/classes/scores/count outputs, whose boxes are normalised `ymin,xmin,ymax,xmax`.'
    boxes, classes, scores = [np.asarray(o, np.float32).reshape(-1, 4 if i == 0 else 1) for i, o in enumerate(outs[:3])]
    n = int(np.asarray(outs[3]).reshape(-1)[0]) if len(outs) > 3 else len(scores)
    keep = [i for i in range(min(n, len(scores))) if scores[i, 0] >= conf]
    b = boxes[keep][:, [1, 0, 3, 2]]                          # ymin,xmin,ymax,xmax -> xyxy
    # normalised against the padded input, so they go through scale_boxes like any other detector's
    h, w = (meta or {}).get('size') or (1, 1)
    box = scale_boxes(b * np.array([w, h, w, h], np.float32), meta) if meta else b
    return [dict(label=label_at(labels, int(classes[i, 0]) + offset), score=round(float(scores[i, 0]), 6),
                 index=int(classes[i, 0]) + offset, box=[round(float(v), 2) for v in box[j]])
            for j, i in enumerate(keep)]

In [ ]:
#| hide
_b = np.array([[[0.1, 0.2, 0.5, 0.6]]], np.float32)
_d = decode_ssd([_b, [[0]], [[0.8]], [1]], ['cat'], meta=dict(orig=(100, 200), size=(100, 200)))
test_eq(_d[0]['label'], 'cat'); test_close(_d[0]['box'], [40., 10., 120., 50.])
# the same box out of a letterboxed 100x100 input: the grey bands come off before the scaling
_d = decode_ssd([_b, [[0]], [[0.8]], [1]], ['cat'],
                meta=dict(orig=(100, 200), size=(100, 100), ratio=0.5, pad=(0, 25)))
test_close(_d[0]['box'], [40., 0., 120., 50.])

In [ ]:
#| export
def decode_detect_auto(outs,            # every output array the model returned, in declared order
                       labels=None,
                       conf:float=0.25,
                       iou:float=0.45,
                       meta:dict=None
                      ) -> list:
    'Decode a detector without being told its family: a postprocess head is SSD, one tensor is YOLO.'
    outs = list(outs)
    if len(outs) == 1: return decode_yolo(outs[0], labels, conf=conf, iou=iou, meta=meta)
    if is_ssd(outs): return decode_ssd(outs, labels, conf=conf, meta=meta)
    raise ValueError(f'{len(outs)} outputs of shape {[list(np.shape(o)) for o in outs[:4]]}: neither a YOLO '
                     'head nor boxes/classes/scores/count. anya does not decode a raw per-stride head; '
                     'export the model with its postprocessing included, or pass a decoder.')

def is_ssd(outs) -> bool:
    'Are these the four tensors of a TFLite Detection PostProcess head: boxes, classes, scores, count?'
    if len(outs) < 3: return False
    s = [np.shape(o) for o in outs[:3]]
    n = s[0][-2] if len(s[0]) >= 2 else 0
    return s[0][-1] == 4 and n > 0 and all(int(np.prod(x)) == n for x in s[1:])

In [ ]:
#| hide
test_eq(decode_detect_auto([np.array([[[0.1,0.2,0.5,0.6]]], np.float32), [[0]], [[0.8]], [1]],
                           ['cat'], meta=dict(orig=(100,200)))[0]['label'], 'cat')
test_eq(len(decode_detect_auto([np.zeros((1, 7, 20), np.float32)], ['a','b','c'], conf=0.5)), 0)
# ssdlite320's tflite export ships twelve raw per-stride heads: an error beats an empty answer
test_fail(lambda: decode_detect_auto([np.zeros((1, 546, 20, 20), np.float32)]*12), contains='raw per-stride')

In [ ]:
#| export
def chan_first(shape,          # a 3-axis segmentation output shape
               n:int=None      # how many classes, when the labels say
              ) -> bool:
    'Is a segmentation output C,H,W rather than H,W,C? SegFormer puts 150 classes on a 128x128 grid.'
    c, h, w = (int(x) for x in shape)
    if n and n in (c, w) and c != w: return c == n
    if h == w != c: return True                    # the last two axes match, so they are the pixel grid
    if h == c != w: return False
    return c < w

def resize_mask(m, size:tuple) -> np.ndarray:
    'Nearest-neighbour resize of a label map, so no class the model never emitted appears in it.'
    from PIL import Image
    im = Image.fromarray(np.asarray(m, np.int32), mode='I')
    return np.asarray(im.resize((size[1], size[0]), Image.NEAREST), dtype=np.int32)

def decode_segment(out,            # (1, C, H, W), (1, H, W, C) or (1, H, W) logits or a label map
                   labels=None,
                   meta:dict=None,
                   min_frac:float=0.005   # ignore classes below this share of pixels
                  ) -> dict:
    'Argmax a segmentation output into a label map plus the per-class pixel share.'
    a = np.asarray(out, np.float32)
    if a.ndim == 4: a = a[0]
    m = a.argmax(0 if chan_first(a.shape, len(labels) if labels else None) else -1) if a.ndim == 3 else a
    m = np.asarray(m, np.int32)
    orig, pad = (meta or {}).get('orig'), (meta or {}).get('pad') or (0, 0)
    # a stretched map covers the whole picture, so it can go back on it; a padded or cropped one cannot
    if orig and not any(pad) and tuple(orig) != m.shape: m = resize_mask(m, orig)
    ids, cnt = np.unique(m, return_counts=True)
    tot = float(m.size)
    cls = [dict(label=label_at(labels, int(i)), index=int(i), frac=round(float(c/tot), 5))
           for i, c in sorted(zip(ids, cnt), key=lambda t: -t[1]) if c/tot >= min_frac]
    return dict(mask=m, shape=list(m.shape), classes=cls)

def mask_image(mask,          # an integer label map
               palette=None   # (n,3) uint8 colours, else a deterministic one
              ) -> np.ndarray:
    'Colour an integer mask for looking at.'
    m = np.asarray(mask, np.int64); n = int(m.max())+1
    if palette is None:
        rng = np.random.default_rng(0)
        palette = np.concatenate([np.zeros((1,3), np.uint8), rng.integers(40, 255, (max(n,1), 3), dtype=np.uint8)])
    return np.asarray(palette, np.uint8)[m % len(palette)]

In [ ]:
#| hide
_s = np.zeros((1, 3, 4, 4), np.float32); _s[0, 2] = 5.
_r = decode_segment(_s, ['bg','road','sky'])
test_eq(_r['classes'][0]['label'], 'sky'); test_eq(_r['classes'][0]['frac'], 1.0)
test_eq(mask_image(_r['mask']).shape, (4,4,3))
test_eq(decode_segment(_s, ['bg','road','sky'], meta=dict(orig=(8, 12), pad=(0,0)))['shape'], [8, 12])
test_eq(chan_first((150, 128, 128), 150), True)   # more classes than pixels across, still channels-first
test_eq(chan_first((128, 128, 150), 150), False)
test_eq(chan_first((21, 224, 224)), True)

In [ ]:
#| export
def l2norm(v, axis=-1):
    'Unit-length rows, so a dot product is a cosine.'
    v = np.asarray(v, np.float32)
    return v / np.maximum(np.linalg.norm(v, axis=axis, keepdims=True), 1e-12)

def similarity(a, b) -> np.ndarray:
    'Cosine similarity between two sets of embeddings.'
    return l2norm(np.atleast_2d(a)) @ l2norm(np.atleast_2d(b)).T

def pool_embed(out) -> np.ndarray:
    'One unit-length vector from an embedding output; a token sequence or feature map is mean-pooled.'
    a = np.asarray(out, np.float32)
    if a.ndim > 1: a = a[0]                                        # the batch axis, always 1 here
    if a.ndim == 3: a = a.mean((1, 2) if chan_first(a.shape) else (0, 1))
    if a.ndim == 2: a = a.mean(0)                                  # 257 patch tokens of a ViT, say
    return l2norm(a.reshape(-1))

In [ ]:
#| hide
test_close(float(similarity([1.,0.], [1.,0.])[0,0]), 1.0)
test_close(float(similarity([1.,0.], [0.,1.])[0,0]), 0.0)
test_eq(pool_embed(np.ones((1, 384), np.float32)).shape, (384,))
test_eq(pool_embed(np.ones((1, 257, 384), np.float32)).shape, (384,))    # a ViT's tokens, pooled
test_eq(pool_embed(np.ones((1, 1280, 7, 7), np.float32)).shape, (1280,)) # a CNN's feature map, pooled

In [ ]:
#| export
def label_map(d:dict) -> L|None:
    'Class names out of an `id2label`, an inverted `label2id`, or a bare index/name map.'
    m = d.get('id2label') or {i: n for n, i in (d.get('label2id') or {}).items()} or d
    try: return L(v for _, v in sorted((int(k), v) for k, v in m.items())) or None
    except (TypeError, ValueError): return None          # a config.json that names no classes

def read_labels(o) -> L|None:
    'Read class names from a labels file, a `config.json` or its contents, or an iterable.'
    if o is None: return None
    if isinstance(o, dict): return label_map(o)
    if isinstance(o, (str, Path)) and Path(o).exists():
        t = Path(o).read_text(encoding='utf-8', errors='replace').strip()
        if t.startswith('{'): return label_map(json.loads(t))
        if t.startswith('['): return L(json.loads(t))
        # a plain labels.txt, optionally "0 tench" or "0:tench" per line
        return L(_strip_idx(l) for l in t.split('\n') if l.strip())
    return L(o)

def _strip_idx(line:str) -> str:
    s = line.strip()
    for sep in (':', ' ', '\t'):
        h, _, t = s.partition(sep)
        if t and h.strip().isdigit(): return t.strip()
    return s

In [ ]:
#| hide
_lp = _td/'labels.txt'; _lp.write_text('0 tench\n1 goldfish\n')
test_eq(read_labels(_lp), ['tench','goldfish'])
_jp = _td/'labels.json'; _jp.write_text('{"1": "b", "0": "a"}')
test_eq(read_labels(_jp), ['a','b'])
test_eq(read_labels(['x','y']), ['x','y'])
test_eq(read_labels(None), None)
test_eq(read_labels({'id2label': {'1': 'b', '0': 'a'}}), ['a','b'])
test_eq(read_labels({'label2id': {'person': 0, 'car': 1}}), ['person','car'])  # how a yolo export ships them
test_eq(read_labels({'architectures': ['Whatever']}), None)                   # a config that names no classes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()